In [64]:
import requests
import time
import pandas as pd
import re
from fake_useragent import UserAgent
from bs4 import BeautifulSoup

In [67]:
# ### 1.1. Функции для парсинга

def get_page(url, headers):
    """Загружает HTML страницы с заданными заголовками."""
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        print(f'Ошибка при запросе {url}: {e}')
        return None

def parse_datetime(dt_str):
    """Преобразует строку даты в datetime с UTC таймзоной."""
    if not dt_str:
        return None
    # Заменяем Z на +00:00 для совместимости с fromisoformat
    dt_str = dt_str.replace('Z', '+00:00')
    dt = datetime.fromisoformat(dt_str)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt

def parse_article_card(card, base_url):
    """
    Извлекает данные из одного блока статьи на странице ленты.
    Возвращает словарь с полями.
    """
    data = {}

    # Заголовок и ссылка
    title_tag = card.find('a', class_='tm-title__link')
    if title_tag:
        data['title'] = title_tag.get_text(strip=True)
        data['url'] = base_url + title_tag['href'] if title_tag['href'].startswith('/') else title_tag['href']
    else:
        data['title'] = None
        data['url'] = None

    # Тип публикации (статья, новость, пост и т.п.)
    type_tag = card.find('span', class_='tm-article-snippet__hubs-item')
    if type_tag:
        data['type'] = type_tag.get_text(strip=True)
    else:
        data['type'] = None

    # Время чтения
    reading_time_tag = card.find('span', class_='tm-article-reading-time__label')
    if reading_time_tag:
        text = reading_time_tag.get_text(strip=True)
        match = re.search(r'(\d+)\s*мин', text)
        data['reading_time_min'] = int(match.group(1)) if match else None
    else:
        data['reading_time_min'] = None

    # Просмотры (находим элемент с иконкой просмотров)
    views_tag = card.find('span', class_='tm-icon-counter__value')
    if views_tag:
        text = views_tag.get_text(strip=True)
        text_clean = re.sub(r'[^\d]', '', text)
        data['views'] = int(text_clean) if text_clean else None
    else:
        data['views'] = None

    # Хаб/тег (возьмём первый)
    hub_tag = card.find('a', class_='tm-hub-link')
    if hub_tag:
        data['hub'] = hub_tag.get_text(strip=True)
    else:
        data['hub'] = None

    # Дата публикации
    time_tag = card.find('time')
    if time_tag:
        datetime_attr = time_tag.get('datetime')
        if datetime_attr:
            data['published'] = parse_datetime(datetime_attr)
        else:
            data['published'] = None
    else:
        data['published'] = None

    # Количество комментариев
    comments_link = card.find('a', class_='tm-comments-counter-link')
    if comments_link:
        text = comments_link.get_text(strip=True)
        match = re.search(r'(\d+)', text)
        data['comments'] = int(match.group(1)) if match else 0
    else:
        data['comments'] = 0

    # Автор (имя или компания)
    author_tag = card.find('a', class_='tm-user-info__username')
    if author_tag:
        data['author'] = author_tag.get_text(strip=True)
    else:
        data['author'] = None

    # Краткое описание
    desc_tag = card.find('div', class_='tm-article-body')
    if desc_tag:
        desc_text = desc_tag.get_text(strip=True)
        data['description'] = desc_text[:500]
    else:
        data['description'] = None

    return data

def parse_feed_page(html, base_url):
    """Парсит HTML страницы ленты и возвращает список словарей с данными статей."""
    soup = BeautifulSoup(html, 'html.parser')
    articles = []

    cards = soup.find_all('article', class_='tm-articles-list__item')
    for card in cards:
        data = parse_article_card(card, base_url)
        if data['url']:
            articles.append(data)

    return articles


In [68]:
# ### 1.2. Сбор данных за июнь 2026
from datetime import datetime, timedelta, timezone
# %%
BASE_URL = 'https://habr.com'
FEED_URL = BASE_URL + '/ru/feed/'

# Целевой месяц: июнь 2026 (используем UTC таймзону)
TARGET_START = datetime(2026, 6, 1, tzinfo=timezone.utc)
TARGET_END = datetime(2026, 7, 1, tzinfo=timezone.utc)

ua = UserAgent()
headers = {'User-Agent': ua.random}

all_articles = []
page = 1
stop = False

while not stop:
    if page == 1:
        url = FEED_URL
    else:
        url = f'{FEED_URL}page{page}/'

    print(f'Загружаем страницу {page}...')
    html = get_page(url, headers)
    if html is None:
        print('Не удалось загрузить страницу, завершаем.')
        break

    articles = parse_feed_page(html, BASE_URL)
    if not articles:
        print('Нет статей на странице, завершаем.')
        break

    # Фильтруем по дате
    for article in articles:
        pub_date = article.get('published')
        if pub_date is None:
            continue
        if pub_date < TARGET_START:
            stop = True
            break
        if pub_date < TARGET_END:  # в июне
            all_articles.append(article)

    if stop:
        print(f'Достигли статей до июня 2026 на странице {page}, останавливаемся.')
        break

    page += 1
    time.sleep(0.3)

print(f'Собрано {len(all_articles)} статей за июнь 2026.')

# %%
df = pd.DataFrame(all_articles)
df.head()

Загружаем страницу 1...
Загружаем страницу 2...
Загружаем страницу 3...
Загружаем страницу 4...
Загружаем страницу 5...
Загружаем страницу 6...
Загружаем страницу 7...
Загружаем страницу 8...
Загружаем страницу 9...
Загружаем страницу 10...
Загружаем страницу 11...
Загружаем страницу 12...
Загружаем страницу 13...
Загружаем страницу 14...
Загружаем страницу 15...
Загружаем страницу 16...
Загружаем страницу 17...
Загружаем страницу 18...
Загружаем страницу 19...
Загружаем страницу 20...
Загружаем страницу 21...
Загружаем страницу 22...
Загружаем страницу 23...
Загружаем страницу 24...
Загружаем страницу 25...
Загружаем страницу 26...
Загружаем страницу 27...
Загружаем страницу 28...
Загружаем страницу 29...
Загружаем страницу 30...
Загружаем страницу 31...
Загружаем страницу 32...
Загружаем страницу 33...
Загружаем страницу 34...
Загружаем страницу 35...
Загружаем страницу 36...
Загружаем страницу 37...
Загружаем страницу 38...
Загружаем страницу 39...
Загружаем страницу 40...
Загружаем

,title,url,type,reading_time_min,views,hub,published,comments,author,description
0,Пользователи WhatsApp** теперь могут зарезерви...,https://habr.com/ru/news/1053676/,None,2,10,None,2026-06-30 23:40:40+00:00,0,darya_kiwi,None
1,Практическое махоботоводство в 2026 году. Част...,https://habr.com/ru/articles/1053638/,None,12,87,None,2026-06-30 22:53:24+00:00,0,lubezniy,None
2,Обзор выставки «Ключ к доверию. Безопасность в...,https://habr.com/ru/articles/1054158/,None,14,97,None,2026-06-30 20:48:08+00:00,0,Lexx_Nimofff,None
3,Разработчик кабелей «Инкаб» разместил акции на...,https://habr.com/ru/news/1054156/,None,2,93,None,2026-06-30 20:46:02+00:00,0,Lexx_Nimofff,None
4,"В AirDrop нашли уязвимости, позволяющие времен...",https://habr.com/ru/news/1054154/,None,2,88,None,2026-06-30 20:43:15+00:00,0,daniilshat,None


In [69]:
# Посмотрим на пропуски данных.

# %%
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   title             95 non-null     str                
 1   url               95 non-null     str                
 2   type              0 non-null      object             
 3   reading_time_min  95 non-null     int64              
 4   views             95 non-null     int64              
 5   hub               0 non-null      object             
 6   published         95 non-null     datetime64[us, UTC]
 7   comments          95 non-null     int64              
 8   author            95 non-null     str                
 9   description       0 non-null      object             
dtypes: datetime64[us, UTC](1), int64(3), object(3), str(3)
memory usage: 7.6+ KB
